# 🦙 Customer Support QLoRA Fine-tune (LLaMA-3.2-3B) — Explained

**What this notebook does:** fine-tunes Meta's **LLaMA-3.2-3B-Instruct** to answer customer-support FAQs, using **QLoRA** = **4-bit quantization** (to fit the model in memory) **+ LoRA** (to train only tiny adapters instead of the whole model).

**The flow:**
`install → login to HF → load LLaMA in 4-bit → load & chat-format the data → add LoRA adapters → train (SFTTrainer) → test → save/upload`

**How it compares to the other project (`customer-support-ticket-tagger`):**
| | Ticket-tagger | This notebook |
|---|---|---|
| Model | DeBERTa-v3-base (~184M, encoder) | LLaMA-3.2-3B (decoder LLM) |
| Method | **Full** fine-tuning | **QLoRA** (4-bit + LoRA) |
| Task | Classify → 1 of 10 departments | **Generate** FAQ answers |

Each code cell below has a plain-English explanation right above it.

---

# Installing necessary libraries

* transformers: Provides state-of-the-art pretrained models for NLP, computer vision, and beyond.
* datasets: A library for easy access to a wide range of datasets for ML and NLP tasks.
* accelerate: Simplifies distributed training and inference for PyTorch models.
* torch: PyTorch library for building and training deep learning models.
* bitsandbytes: Optimized GPU quantization and acceleration for large-scale models.
* peft: Parameter-efficient fine-tuning techniques for large language models.
* trl: Tools for training transformer models with reinforcement learning techniques.

### 📦 Install the exact libraries (pinned versions)

Installs the packages with **specific version numbers** (e.g. `transformers==4.47.1`).
- Why pin versions? So the notebook **always behaves the same** and doesn't break when libraries update (exactly the kind of breakage you hit earlier with `eval_strategy`).
- Key ones: `transformers` (models), `bitsandbytes` (4-bit quantization), `peft` (LoRA), `trl` (the `SFTTrainer`), `datasets`, `accelerate`.

In [ ]:
!pip install transformers==4.47.1 accelerate==0.34.2 bitsandbytes==0.45.0 trl==0.13.0 datasets==3.2.0 peft==0.14.0 tokenizers==0.21.0 huggingface_hub==0.26.0

# Importing Libraries

### 🧰 Import the tools

- `AutoModelForCausalLM` → loads a **decoder/generative LLM** (LLaMA is a decoder)
- `AutoTokenizer`, `LlamaTokenizer` → text ↔ tokens
- `BitsAndBytesConfig` → the **4-bit quantization** settings (the "Q" in QLoRA)
- `torch` → the math engine
- `load_dataset` → grab data from the Hugging Face Hub
- `LoraConfig`, `get_peft_model`, `PeftModel` → **LoRA** (the "efficient fine-tuning" part)
- `SFTTrainer`, `SFTConfig` (from `trl`) → **Supervised Fine-Tuning trainer**, built for instruction-tuning LLMs

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, LlamaTokenizer, BitsAndBytesConfig
import torch
from datasets import load_dataset
from transformers import Trainer, TrainingArguments
from peft import PeftModel,get_peft_model,LoraConfig, TaskType
from trl import SFTTrainer, SFTConfig

### 🔑 Log in to Hugging Face (gated model)

LLaMA is a **gated model** — you must be logged in + have accepted Meta's license.
- `UserSecretsClient()` → reads your saved secret from **Kaggle Secrets** (named "HuggingFace")
- `login(token=hf_token)` → logs you in so you can download LLaMA
- (Same idea as an Access Token — kept secret, not hardcoded.)

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HuggingFace") # Fetching the Hugging Face token from the Kaggle Secret keys add on
login(token = hf_token) # Logging into Hugging Face Hub to access models and other resources

# Loading Model configurations and Dataset Preparation

Huggingface model link: https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct

### 🦙 Pick the model

`meta-llama/Llama-3.2-3B-Instruct` = Meta's **LLaMA 3.2**, **3-billion-parameter**, **instruction-tuned** (chat-following) model. This is a real **decoder LLM** — much bigger than the DeBERTa (184M) from the other project.

In [ ]:
base_model = 'meta-llama/Llama-3.2-3B-Instruct'

### 📉 Load the model in 4-bit — the "Q" in QLoRA

`BitsAndBytesConfig` sets up **4-bit quantization** so the 3B model fits in limited GPU memory:
- `load_in_4bit=True` → store the model in **4-bit** (tiny)
- `bnb_4bit_quant_type='nf4'` → the smart **NF4** format (keeps accuracy)
- `bnb_4bit_compute_dtype=torch.float16` → do the *math* in 16-bit
- `bnb_4bit_use_double_quant=True` → **double quantization** (extra squeeze for stability)

Then `AutoModelForCausalLM.from_pretrained(..., quantization_config=bnb_config, device_map="auto")` loads LLaMA **in 4-bit**, auto-spread across your GPU.

👉 This is the **quantization** half of **QLoRA**. (LoRA comes in cell for `LoraConfig`.)

In [ ]:
# Configure 4-bit quantization settings using the BitsAndBytesConfig class
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # Enable loading the model with 4-bit precision for reduced memory usage
    bnb_4bit_quant_type='nf4',  # Use NormalFloat4 (nf4), a quantization format for higher accuracy
    bnb_4bit_compute_dtype=torch.float16,  # Use float16 for computation to balance speed and precision
    bnb_4bit_use_double_quant=True  # Enable double quantization for better numerical stability
)

# Load the pre-trained model with 4-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    base_model,  # Name of the base model defined earlier
    device_map="auto",  # Automatically map model layers to available devices (e.g., GPU/CPU)
    quantization_config=bnb_config,  # Apply the defined 4-bit quantization configuration
)

# Note:
# 1. The use of 4-bit quantization helps in reducing memory requirements while maintaining reasonable performance.
# 2. `device_map="auto"` ensures the model layers are automatically distributed across available hardware for efficient loading.

### ✂️ Load & configure the tokenizer

- Load LLaMA's tokenizer
- `pad_token = eos_token` → use the end-of-sentence token as padding filler
- `padding_side = "right"` → add padding on the right
(Housekeeping so batches line up — same as the other notebook.)

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
# Set the padding token to the end-of-sequence (eos) token
# This ensures compatibility when the model processes inputs with padding
tokenizer.pad_token = tokenizer.eos_token
# Configure the tokenizer to apply padding on the right side of the input
# This is often the default for causal language models to ensure alignment during training or inference
tokenizer.padding_side = "right"

Dataset link: https://huggingface.co/datasets/Victorano/customer-support-1k

### 📊 Load & split the dataset

- `load_dataset("Victorano/customer-support-1k", split="train")` → download a **1,000-example customer-support dataset** from the HF Hub
- `remove_columns([...])` → drop columns we don't need, keeping just the question/response
- `train_test_split(test_size=0.2)` → 80% train / 20% test

In [ ]:
# Loading the 'Customer_support_faqs_dataset' from the Hugging Face dataset repository
dataset = load_dataset("Victorano/customer-support-1k", split="train")
dataset = dataset.remove_columns(['flags', 'category','intent','text'])
dataset = dataset.train_test_split(test_size=0.2)

### 👀 Peek at the dataset

Just displays the dataset structure — the train/test splits and their row counts/columns.

In [ ]:
dataset

### 📝 Format each example as a chat (the key data-prep step)

- `instruction` = the **system prompt** — the bot's personality/rules (be concise, friendly, escalate if unsure)
- `template(row)` builds a 3-message chat for each row: **system** (instruction) + **user** (the question) + **assistant** (the correct answer)
- `tokenizer.apply_chat_template(..., tokenize=False)` → turns that into the exact chat-format text LLaMA expects, stored in a new `text` column
- `dataset.map(template, num_proc=4)` → apply to every row (4 parallel processes for speed)

This teaches the model: *"given this question, produce this answer, in this style."*

In [ ]:
# Defining the instruction that will guide the assistant's behavior for providing customer support answers.

instruction = """You are a helpful and efficient customer support bot designed to assist users by providing answers to frequently asked questions (FAQs) related to our products and services. Your responses should be concise, clear, and friendly, ensuring the user feels heard and supported. If the user’s question is outside the scope of the FAQ, gently direct them to contact customer support.

Always prioritize accuracy and clarity in your answers.
If the user asks a complex question, break it down into smaller, manageable parts and answer step-by-step.
Provide useful links or references to detailed documentation when appropriate.
Use a friendly and professional tone, ensuring the response is easy to understand.
If the FAQ does not cover the question, offer an apology and suggest contacting customer support.
"""

def template(row):
    # Creating a list of message exchanges (system, user, assistant)
    row_json = [{"role": "system", "content": instruction }, # System message with the pre-defined instructions
               {"role": "user", "content": row["instruction"]}, # User's question from the dataset
               {"role": "assistant", "content": row["response"]}] # The assistant's answer from the dataset

    # Tokenizing the chat template and storing the result in the 'text' column (without applying tokenization)
    row["text"] = tokenizer.apply_chat_template(row_json, tokenize=False)
    return row

# Applying the template function to each row in the dataset using multi-processing (4 processes in parallel)
dataset = dataset.map(template,num_proc= 4)

### 👀 Check one formatted example

Prints one training row's `text` so you can see the final chat-formatted string the model will actually learn from.

In [ ]:
# To check a sample record from dataset
dataset['train']['text'][10]

### ⚡ Set up LoRA — the "LoRA" in QLoRA

`LoraConfig` defines the small trainable adapters (instead of training the whole 3B model):
- `r=4` → **rank** — size of the tiny low-rank matrices (small = fewer params)
- `lora_alpha=8` → a **scaling factor** for the adapter's effect
- `lora_dropout=0.2` → dropout for regularization (prevents overfitting)
- `task_type="CAUSAL_LM"` → text-generation task

- `get_peft_model(model, lora_config)` → **wraps** the 4-bit LLaMA with LoRA adapters; the base stays **frozen**, only the adapters train
- `model.print_trainable_parameters()` → shows how **few** params are actually trained (often <1% of the total!) — the whole point of LoRA

👉 4-bit base (quantization) + LoRA adapters = **QLoRA**.

In [ ]:
# Configure LoRA (Low-Rank Adaptation) for fine-tuning the model on a language modeling task
lora_config = LoraConfig(
    r=4,                   # Rank for low-rank matrices
    lora_alpha=8,         # Scaling factor
    lora_dropout=0.2,      # Regularization dropout
    task_type="CAUSAL_LM"  # For language modeling
)
model = get_peft_model(model, lora_config)
# Print the number of trainable parameters in the model after applying LoRA
model.print_trainable_parameters()

### ⚙️ Training settings + the SFTTrainer

`TrainingArguments` — the training knobs:
- `num_train_epochs=1` → go through the data once
- `per_device_train_batch_size=1` → 1 example at a time (a 3B model is memory-heavy)
- `warmup_steps=5` → ease the learning rate up at the start
- `learning_rate=2e-4` → step size
- `fp16=True` → 16-bit precision for speed/memory
- `report_to="none"` → no external logging

`SFTTrainer` (from `trl`) = **Supervised Fine-Tuning trainer**, made for instruction-tuning LLMs. It gets the model, train/test data, tokenizer, args, and `peft_config=lora_config` (the LoRA setup).

*(Minor note: LoRA is applied both via `get_peft_model` above and `peft_config` here — slightly redundant, but harmless.)*

In [ ]:
# Setting up training arguments for the model training process
training_arguments = TrainingArguments(
    output_dir="./results",  # Directory where the results will be saved
    num_train_epochs=1,  # Number of training epochs
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    warmup_steps=5,  # Number of warmup steps to gradually increase the learning rate during training
    learning_rate=2e-4,  # Learning rate for the optimizer
    fp16=True,  # Enabling 16-bit floating point precision for faster training on GPUs that support it (reduces memory usage)
    report_to="none",  # Disabling logging/reporting to external services (e.g., TensorBoard, Weights & Biases)
)

# Initializing the SFTTrainer for supervised fine-tuning
trainer = SFTTrainer(
    model=model,  # The pre-trained model to be fine-tuned
    train_dataset=dataset["train"], # The dataset used for training
    eval_dataset=dataset["test"],  # The dataset used for validation
    tokenizer=tokenizer,  # Tokenizer to process input text for the model
    args=training_arguments,  # The training arguments defined above
    peft_config=lora_config,
)

### 🏋️ Train (the actual QLoRA fine-tuning)

- `model.train()` → put the model in **training mode**
- `trainer.train()` → **the learning happens here.** Only the small LoRA adapters get updated (the 4-bit base LLaMA stays frozen), which is what makes fine-tuning a 3B model possible on modest hardware.

In [ ]:
model.train()
trainer.train()

# Testing

### 🧪 A function to test the fine-tuned model

`generate(input_prompt)` runs the full chat flow (same pattern as the BERT doc's `predict`):
1. Build messages: **system** (instruction) + **user** (your question)
2. `apply_chat_template(..., add_generation_prompt=True)` → format + add the "assistant's turn" cue
3. Tokenize → move to GPU (`.to("cuda")`)
4. `model.generate(..., max_new_tokens=2048)` → produce the answer
5. `tokenizer.decode(..., skip_special_tokens=True)` → numbers back to clean text

In [ ]:
# Function to generate a response based on the input prompt
def generate(input_prompt):
    # Define the system and user messages to provide context for the conversation
    messages = [
        {"role": "system", "content": instruction},  # System message with the pre-defined instructions
        {"role": "user", "content": input_prompt}   # User's input prompt
    ]

    # Apply the chat template to format the messages, without tokenizing yet, and add the generation prompt
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Tokenize the formatted prompt, padding and truncating as necessary, and move the data to the GPU
    inputs = tokenizer(prompt, return_tensors='pt', padding=True, truncation=True).to("cuda")

    # Generate the model's output based on the tokenized input, limiting to a maximum of 2048 new tokens
    outputs = model.generate(**inputs, max_new_tokens=2048, num_return_sequences=1)

    # Decode the output tokens back into text, skipping any special tokens (like padding)
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return text  # Return the generated text

### ▶️ Ask it a real question

Calls `generate(...)` with a sample customer question and prints the model's full response.

In [ ]:
response = generate("Where to see what payment options are available?")
print(response)

### 🧹 Clean up the output

The raw output includes the whole prompt (system + user + assistant). `response.split("assistant")[-1]` grabs just the part **after** "assistant" — i.e. only the model's actual reply.

In [ ]:
# to format output
print(response.split("assistant")[-1])

# Save the model

### 💾 Save the fine-tuned model

`model.save_pretrained(...)` saves the result. Because this is a **PEFT/LoRA** model, it saves just the **small LoRA adapters** (a few MB), not a full copy of LLaMA.
*(Note: the `/content/...` path is a Google-Colab folder; on Kaggle you'd use a different path like `/kaggle/working/...`.)*

In [ ]:
model.save_pretrained("/content/customer-faq-llama-3.2-3B") # Saves the model under the same directory.

### ☁️ Upload to the Hugging Face Hub

`model.push_to_hub("customer-faq-llama-3.2-3B")` uploads your trained adapters to **your** Hugging Face account so others (or future-you) can download and use them. (Requires being logged in — done in the login cell.)

In [ ]:
# To push the model to hugginface
model.push_to_hub("customer-faq-llama-3.2-3B")

Model would be saved like this: https://huggingface.co/Chirag4579/customer-faq-llama-3.2-3B

# Load a model

### 📥 (Reference) How to load the model later

This cell is **commented out** — it just shows how you'd **reload** the fine-tuned model later, either:
- **locally** from the saved folder, or
- **from the Hugging Face Hub** (with the same 4-bit `bnb_config`).
Nothing runs here; it's a copy-paste template for reuse.

In [ ]:
# load locally
# local_model = AutoModelForCausalLM.from_pretrained("/content/customer-faq-llama-3.2-3B")

# load the saved model from huggingface

# model = AutoModelForCausalLM.from_pretrained(
#     "Chirag4579/customer-faq-llama-3.2-3B",  # Name of the base model defined earlier
#     device_map="auto",  # Automatically map model layers to available devices (e.g., GPU/CPU)
#     quantization_config=bnb_config,  # Apply the defined 4-bit quantization configuration
# )

---

# ❓ Q&A Notes

## Q1: What does "state-of-the-art pretrained models" mean? Does it mean we get pre-trained models so we don't have to train them?

### "State-of-the-art" (SOTA)
**= the best / most advanced available right now.** So "state-of-the-art pretrained models" = "the most advanced, best-performing models currently available" (LLaMA, BERT, etc.) — not toy models. *(Everyday meaning: like a phone being "state-of-the-art" = newest and best.)*

### "Pretrained" — yes, your instinct is right ✅
**Pretrained = already trained** (by the model's creators) on massive data. So you **download a ready-made model and don't train it from scratch.** Training from scratch would need enormous datasets, millions in GPUs, and weeks/months — pretrained saves all that.

### One important nuance: two levels of use

| You want to... | Do you train? |
|---|---|
| **Just use it as-is** (e.g. `pipeline('sentiment-analysis')`) | ❌ **No training** — use directly |
| **Adapt it to your data** (fine-tuning) | ⚠️ **A little** — you *fine-tune* (like this notebook), but still **not from scratch** |

- ✅ You **never train from scratch** — pretrained saves you that.
- You *may* choose to **fine-tune** (small extra training on your data) to specialize it — exactly what this QLoRA notebook does.

### The sentence decoded
*"state-of-the-art pretrained models for NLP, computer vision, and beyond"* =
- **state-of-the-art** → best/most advanced
- **pretrained** → already trained, ready to download & use
- **for NLP, computer vision, and beyond** → text, images, and more

**One line:** "State-of-the-art" = the best/most advanced models; "pretrained" = already trained on huge data, so you download and use them without training from scratch. You can still optionally fine-tune one on your own data (like here), but you never start from zero.

## Q2: What is `trl` here?

**TRL = Transformer Reinforcement Learning** — a Hugging Face library for **training/fine-tuning LLMs** (instruction-tuning and alignment).

### In simple terms
A **toolbox built on top of `transformers`** with ready-made trainers:

| Tool | What it does |
|---|---|
| **`SFTTrainer`** ⭐ | **Supervised Fine-Tuning** — train on example question→answer pairs (what this notebook uses) |
| **`SFTConfig`** | The settings for that trainer |
| `DPOTrainer`, `PPOTrainer` | Advanced **alignment** (learn from human preferences — how chat models get polished) |

### Why this notebook uses it
It uses **`SFTTrainer`** to do **supervised fine-tuning**: showing LLaMA lots of *(customer question → correct answer)* examples so it learns that style. `SFTTrainer` handles the training-loop details and works with LoRA via `peft_config`.

### How it relates to what you know
```
transformers  → load & run models
peft          → LoRA (efficient fine-tuning)
trl           → the trainer (SFTTrainer) that does the training
```
`transformers` + `peft` + `trl` = the trio powering this QLoRA fine-tune.

### Name nuance
Despite "**R**einforcement **L**earning" in the name, TRL now covers **more than RL** — including plain supervised fine-tuning (`SFTTrainer`), which is what you use here (no RL involved).

**One line:** TRL (Transformer Reinforcement Learning) is a Hugging Face library for fine-tuning/aligning LLMs; this notebook uses its `SFTTrainer` to train LLaMA on question→answer examples — despite the name, it does plain supervised fine-tuning too, not just RL.

## Q3: We already have the model (transformers) and the trainer (trl) — so why do we need `peft`?

**`peft` is the piece that makes it LoRA (the "efficient" part).**

Without `peft`, the trainer (`SFTTrainer`) would train the **entire** 3-billion-parameter model — **full fine-tuning** (huge memory, big GPUs). `peft` changes *what* gets trained:
1. **Injects the tiny LoRA adapters** into the model
2. **Freezes** the giant base model (billions of params → not trained)
3. So only the **small adapters** (<1% of params) get updated

### Each library has a *different* job

| Library | Its job | Answers... |
|---|---|---|
| **transformers** | Loads the model | **WHAT** model? (LLaMA) |
| **peft** | Adds LoRA adapters + freezes the base | **WHICH parts** get trained? (only tiny adapters) |
| **trl (`SFTTrainer`)** | Runs the training loop | **HOW** to train? (the process) |

transformers + trl alone = a model + a trainer = **full fine-tuning**. `peft` shrinks *what* the trainer actually updates.

### Analogy 🏗️
Renovating a skyscraper:
- **transformers** = the building (model)
- **trl** = the construction crew (does the work)
- **peft** = the decision to **only renovate a few rooms, not the whole tower** (cheap & fast)

Without `peft`, the crew renovates the *entire* tower (expensive, may not fit your GPU).

### If you remove `peft`?
- Still trains ✅ — but as **full fine-tuning** of all 3B params
- Likely **runs out of GPU memory** on a normal machine ❌
- That's the whole reason LoRA/`peft` exists — to make big-model fine-tuning possible on modest hardware

**One line:** `peft` is what actually makes it LoRA — it injects small trainable adapters and freezes the huge base, so the trainer updates <1% of the parameters. Without it you'd train the whole 3B model (full fine-tuning), which usually won't fit in memory. transformers = the model, trl = how to train, peft = which tiny part to train (the efficiency).

## Q4: What is "instruction-tuned"?

**Instruction-tuned = the model was taught to *follow instructions and answer*, not just continue text.**

- **Base model** → only predicts the next word. Ask *"Capital of France?"* → it might reply *"Capital of Spain? Capital of Italy?"* (just continues the pattern) ❌
- **Instruct model** → trained on lots of *(question → answer)* examples, so it actually replies *"Paris."* ✅

The **"Instruct"** in `Llama-3.2-3B-Instruct` = this version already knows how to chat/answer — a perfect starting point for a support bot.

**One line:** Instruction-tuned = trained to answer/obey instructions instead of just continuing text.

## Q5: Are there other libraries beyond the trio? And what is "quant type"?

### Part 1: Other libraries (the backstage crew)
The `transformers + peft + trl` "trio" was a simplification. The notebook also uses supporting libraries:

| Library | Its job (simple) |
|---|---|
| **transformers** ⭐ | Load & run the model |
| **peft** ⭐ | LoRA (efficient fine-tuning) |
| **trl** ⭐ | The trainer (SFTTrainer) |
| **torch** | The math engine (PyTorch) |
| **bitsandbytes** | Does the **4-bit quantization** (the "Q" in QLoRA) |
| **datasets** | Load & prepare the training data |
| **accelerate** | Run training efficiently across GPU(s) |
| **huggingface_hub** | Log in + download/upload models |

The **trio** does the headline work; **torch + bitsandbytes + datasets + accelerate + huggingface_hub** are the backstage crew. Note **bitsandbytes** actually does the quantization — so really QLoRA = transformers + peft + trl + **bitsandbytes**.

### Part 2: What is "quant type"? (`bnb_4bit_quant_type='nf4'`)
**"Quant type" = the *format* used to store each number when shrinking it to 4 bits.**

| Quant type | What it is |
|---|---|
| **`nf4`** (NormalFloat4) ⭐ | A smart 4-bit format **designed for AI weights** — matches how weights are naturally distributed → keeps **more accuracy** |
| **`fp4`** (Float4) | A plain 4-bit float — works, but usually **less accurate** |

This notebook uses **`nf4`** (the recommended, most accurate option).

**Analogy 🎨:** Repaint a detailed photo using only 16 colors (4-bit = 16 values). `fp4` = 16 evenly-spaced colors; `nf4` = 16 colors *smartly chosen* to match the photo → looks much closer to the original. `nf4` picks its 16 "buckets" to match how weights are really distributed → less quality lost.

**One line:** Beyond the trio, the notebook also uses `torch`, `bitsandbytes` (the actual 4-bit quantizer), `datasets`, `accelerate`, and `huggingface_hub`. And "quant type" (`nf4`) is the *format* for packing numbers into 4 bits — `nf4` (NormalFloat4) is a smart, accuracy-preserving choice for AI weights, better than plain `fp4`.

## Q6: Explain `bnb_4bit_compute_dtype=float16` and `bnb_4bit_use_double_quant=True` in simpler terms

### 1️⃣ `bnb_4bit_compute_dtype=torch.float16` — "store small, compute bigger"

Key idea: **the model is stored in 4-bit, but the actual math is done in 16-bit.**
- **4-bit storage** → saves memory (the model *fits* on your GPU) 💾
- **4-bit is too rough for accurate math** — calculations would lose precision
- So when calculating, each number is temporarily **expanded to 16-bit**, math is done accurately, results stay good ✅

> 🍫 Analogy: keep chocolate **frozen** to save fridge space (4-bit storage), but **let it warm up** before eating so it tastes right (16-bit compute).

So `compute_dtype=float16` = "even though the model is squished to 4-bit, do the arithmetic in 16-bit for accuracy."

### 2️⃣ `bnb_4bit_use_double_quant=True` — "quantize the quantization"

When you quantize, you also produce **extra helper numbers** (*quantization constants*) that record *how* each chunk was shrunk. **Double quantization = shrink those helper numbers too.**
- A **second, small round** of compression on the leftover bookkeeping numbers
- Saves a little **extra memory** (~0.4 bits/parameter) with basically no accuracy loss

> 🗜️ Analogy: zip a folder (first quantization), then notice the **index/notes** about the zip are also biggish, so **zip those too** (double quantization).

### The whole 4-bit config together
```
load_in_4bit=True           → store the model in 4-bit (memory savings)
bnb_4bit_quant_type='nf4'   → use the smart nf4 format (accuracy)
bnb_4bit_compute_dtype=fp16 → but do the math in 16-bit (accurate calculations)
bnb_4bit_use_double_quant   → also compress the compression info (a bit more savings)
```
Together = **maximum memory savings while keeping accuracy** = the point of QLoRA.

**One line:** `compute_dtype=float16` = the model is *stored* in tiny 4-bit but the *math* runs in 16-bit so calculations stay accurate (store small, compute bigger); `use_double_quant=True` = also compress the little "how-we-shrank-it" helper numbers — a second squeeze that saves a bit more memory with no real accuracy loss.

## Q7: What are `pad_token` and `padding_side`, and why change the tokenizer instead of using it as-is?

### First: what is padding?
When training, the model processes **several texts at once (a batch)** — but texts have **different lengths**. They must be the **same length** to process together. **Padding = adding filler tokens** to short ones:
```
"Hi there"          → "Hi there [PAD] [PAD] [PAD]"
"How do I reset it" → "How do I reset it"
```
`[PAD]` = meaningless filler, just to equalize lengths.

### `pad_token` = *which* token is the filler
`tokenizer.pad_token` sets what the filler is.
**Problem:** LLaMA's tokenizer **doesn't come with a pad token** (chat/decoder models often don't). Padding without one → **error**.
**Fix:**
```python
tokenizer.pad_token = tokenizer.eos_token
```
= "use the **end-of-sequence** token (already exists) as filler." Reusing an existing token — inventing nothing.

### `padding_side` = *which side* to add filler
```
right padding:  "Hi there [PAD] [PAD]"   ← filler on the right (standard for training)
left padding:   "[PAD] [PAD] Hi there"
```

### Why change it — why not use it as-is?
Because **as-is it's missing the pad token**, and you **can't do batched training/padding without one.** It's a **required fix, not an optional tweak.**

⚠️ Important: this is **NOT changing the vocabulary** (your earlier worry):
- ✅ Just **setting a couple of config options** (which token = filler, which side)
- ❌ **Not** adding/removing words or altering how real text is tokenized

> 🅿️ Analogy: A parking lot exists but hasn't marked "visitor parking." You don't rebuild the lot — you just **designate an existing spot** as visitor parking (`pad_token = eos_token`) and decide cars park nose-in (`padding_side = "right"`).

### The necessity, in short
| Setting | Why it's needed |
|---|---|
| `pad_token = eos_token` | LLaMA has **no pad token** by default → padding would error → assign one |
| `padding_side = "right"` | Tells it **where** to put filler (right = standard for training) |

**One line:** `pad_token` = which filler token equalizes text lengths for batching; `padding_side` = which side (right) to add it. We set them because LLaMA's tokenizer ships **without** a pad token, so batched training would error — a required config fix (reusing the existing eos token), not a change to the vocabulary or how real text is tokenized.

## Q8: So because QLoRA involves training, sentences of different sizes must be equal-length, so we pad — and pad_token is the token added. Correct?

**Yes — correct!**

```
QLoRA fine-tuning → training happens
   → sentences are different lengths
   → but a batch needs equal lengths
   → so we PAD the short ones with filler
   → pad_token = which token is used as that filler
```

### Two tiny precisions
1. `pad_token` **defines *which* token is the filler** (we chose the eos token). The *padding process* actually *adds* the filler; `pad_token` just says *what* it is.
2. Lengths must be equal **within a batch** (the group processed together) — not every sentence in the whole dataset to one fixed length.

### Cause → effect
Because we're *training* (QLoRA) → we process batches → batches need equal lengths → padding → needs a pad_token. (If only doing a single prediction, padding barely matters — it's the **batched training** that makes it necessary.)

**One line:** Correct — QLoRA involves training, so different-length sentences get batched and padded to equal length; `pad_token` defines *which* token is the filler (here, eos). Just note lengths match *within a batch*, and pad_token *names* the filler while padding *adds* it.

## Q9: So the EOS token defines which token needs padding, and the direction is decided by padding_side. Correct?

**Almost — the direction part is spot-on; one fix on the EOS part.**

### Small correction on EOS
- ❌ Not: "EOS defines *which token needs* padding"
- ✅ Yes: "EOS is the token we pad **WITH** — the filler itself"

`pad_token = eos_token` means: *"When I need filler, **use the EOS token as** that filler."* So EOS = **what the padding is made of**, not a decision about *which* text needs padding. (Which texts need padding = automatic: any text **shorter** than the longest in the batch gets filler.)

### The direction part — correct ✅
```
right → "Hi there [EOS] [EOS]"   (filler on the right)
left  → "[EOS] [EOS] Hi there"
```

| Piece | Correct meaning |
|---|---|
| `pad_token = eos_token` | The EOS token is used **as the filler** we pad with |
| `padding_side = "right"` | The **direction** filler is added (right side) |

**One line:** Almost — `pad_token = eos_token` means the EOS token is used *as the filler* (what we pad **with**), not "which token needs padding." And yes ✅ — `padding_side` decides the **direction** (right) the filler is added.